# 0. 데이터(가장 중요) => LLM의 핵심

In [1]:
text = """
<SOS> <USER> 안녕 <BOT> 안녕하세요 <EOS>
<SOS> <USER> 오늘 날씨 어때 <BOT> 날씨가 좋다 <EOS>
<SOS> <USER> 너 뭐해 <BOT> 공부하고 있다 <EOS>
<SOS> <USER> 밥 먹었어 <BOT> 아직 안 먹었다 <EOS>
<SOS> <USER> 이름이 뭐야 <BOT> 나는 인공지능이다 <EOS>
"""

# 1. 토크나이저 (OOV 방지 포함)

In [2]:
tokens = text.split()

vocab = sorted(list(set(tokens)))
UNK = "<UNK>"

if UNK not in vocab:
    vocab.append(UNK)

stoi = {w:i for i,w in enumerate(vocab)}
itos = {i:w for w,i in stoi.items()}

def encode(s):
    return [stoi.get(w, stoi[UNK]) for w in s.split()]

def decode(l):
    return " ".join([itos[i] for i in l])

# 2. 데이터셋 만들기(학습 데이터)

In [3]:
import torch

seq_len = 8

data = encode(text)

X, y = [], []

for i in range(len(data) - seq_len):
    X.append(data[i:i+seq_len])
    y.append(data[i+1:i+seq_len+1])

X = torch.tensor(X)
y = torch.tensor(y)

# 3. GPT모델

In [4]:
import torch.nn as nn
import torch.nn.functional as F

def causal_mask(size):
    return torch.tril(torch.ones(size, size))

class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape

        Q = self.q(x)
        K = self.k(x)
        V = self.v(x)

        scores = Q @ K.transpose(-2, -1) / (D ** 0.5)

        mask = causal_mask(T).to(x.device)
        scores = scores.masked_fill(mask == 0, float('-inf'))

        attn = F.softmax(scores, dim=-1)

        return attn @ V

class Block(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = SelfAttention(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model*4),
            nn.ReLU(),
            nn.Linear(d_model*4, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.ln1(x + self.attn(x))
        x = self.ln2(x + self.ffn(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, seq_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(seq_len, d_model)

        self.blocks = nn.Sequential(
            Block(d_model),
            Block(d_model)
        )

        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, T = x.shape

        tok = self.token_emb(x)
        pos = self.pos_emb(torch.arange(T, device=x.device))

        x = tok + pos
        x = self.blocks(x)
        x = self.ln(x)

        return self.head(x)

# 4. 학습

In [5]:
model = GPT(len(vocab), d_model=64, seq_len=seq_len)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(300):
    optimizer.zero_grad()

    out = model(X)

    loss = criterion(out.view(-1, len(vocab)), y.view(-1))

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(epoch, loss.item())

/Users/pc/miniforge3/envs/research/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


0 3.319084405899048
50 0.26156485080718994
100 0.1165262833237648
150 0.10211282223463058
200 0.096593476831913
250 0.09370922297239304


# 5. 생성(ChatGPT 핵심 부분) => temperature, Top-k, Top-p중에 temperature샘플링 추가

In [6]:
import torch.nn.functional as F

def generate(model, prompt, max_len=20, temperature=1.0):
    model.eval()

    tokens = encode(prompt)

    for _ in range(max_len):
        x = torch.tensor(tokens[-seq_len:]).unsqueeze(0)

        out = model(x)
        logits = out[0, -1] / temperature

        probs = F.softmax(logits, dim=0)

        next_token = torch.multinomial(probs, 1).item()

        tokens.append(next_token)

        if itos[next_token] == "<EOS>":
            break

    return decode(tokens)

# 6. 실제 대화 실해(User 부분)

In [ ]:
prompt = "<SOS> <USER> 오늘 날씨 어때 <BOT>"

print(generate(model, prompt, temperature=0.8))

## 실시간 채팅 버전

In [7]:
def chat(model):
    print("챗봇 시작 (종료: exit 입력)")

    while True:
        user_input = input("User: ")

        if user_input.lower() == "exit":
            break

        prompt = f"<SOS> <USER> {user_input} <BOT>"

        output = generate(model, prompt, temperature=0.8)

        # <BOT> 이후만 잘라서 출력
        response = output.split("<BOT>")[-1]
        response = response.replace("<EOS>", "").strip()

        print("Bot:", response)

In [8]:
chat(model)

챗봇 시작 (종료: exit 입력)


User:  안녕


Bot: 안녕하세요


User:  너 이름이 뭐니?


Bot: 나는 인공지능이다


...

Bot: 아직 안 먹었다


User:  너는 chat gpt를 알아?


Bot: 안녕하세요


KeyboardInterrupt: Interrupted by user

User:  exit
